### CD4 T-cell Pseudobulk Generation for Differential Expression Analysis
*Dataset: PICA Batch001–Batch007 (PICA0001–PICA0128)* \
*Analysis: Donor-level pseudobulk count generation*

#### Aim
This notebook generates donor-level pseudobulk count matrices from the annotated CD4 T-cell dataset.

Single-cell counts are aggregated at the donor-by-cell-type level to create pseudobulk expression profiles suitable for downstream differential expression analysis using DESeq2.

*Note: This notebook performs pseudobulk generation only. Differential expression testing and gene set enrichment analyses were performed separately in R.*

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from scipy import sparse
from pathlib import Path

In [2]:
adata_cd4 = sc.read_h5ad("/scratch/user/s4575250/BIOX7014_Thesis/write/03_batch_expression/PICA_Batch001-Batch007/PICA_Batch001-Batch007_cd4_combined_annot_with_age_scvi.h5ad")

In [3]:
adata_cd4

AnnData object with n_obs × n_vars = 596486 × 38606
    obs: 'status', 'assignment', 'pica_id', 'pool_id', 'sequencing_batch', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'status_manual', 'cell_type', 'broad_cell_type', 'pica_broad_cell_type', 'pica_cell_type', 'pica_cell_type_01_myeloid', 'pica_cell_type_02_b', 'pica_cell_type_03_nk_t', 'pica_cell_type_complete', 'pica_broad_cell_type_complete', 'pica_broad_cell_type_complete_01_gdT_corrected', 'pica_cell_type_complete_01_gdT_corrected', 'status_manual_01_pica', 'Record ID', 'Sex', 'Age_years', 'Age', 'Event Name', 'Was baseline blood sample collected?', 'IFC ID', "Child's sex", 'Age in 

In [ ]:
# Define metadata columns required for pseudobulk aggregation
donor_col = "pica_id"
celltype_col = "pica_cell_type_complete_01_gdT_corrected"
age_col = "Age_years"
sex_col = "Sex"
batch_col = "sequencing_batch"

In [ ]:
# Check dataset dimensions and CD4 cell-type counts before pseudobulk generation
print(adata_cd4.shape)
print(adata_cd4.obs[celltype_col].value_counts())

(596486, 38606)
pica_cell_type_complete_01_gdT_corrected
CD4 naive/Tcm    407516
CD4 Tem           96510
Tfh               45002
Treg              39431
CD4 cytoT          8027
Name: count, dtype: int64


In [ ]:
# Retain annotated CD4 subsets required for pseudobulk analysis
obs = adata_cd4.obs.copy()

cd4_mask = obs[celltype_col].astype(str).isin([
    "CD4 naive/Tcm",
    "CD4 Tem",
    "CD4 cytoT",
    "Tfh",
    "Treg"
])

# Keep cells with complete donor, cell-type and covariate metadata
required_cols = [donor_col, celltype_col, age_col, sex_col, batch_col]
valid_mask = obs[required_cols].notna().all(axis=1)

keep_mask = cd4_mask & valid_mask

obs_cd4 = obs.loc[keep_mask].copy()

print(obs_cd4.shape)
print(obs_cd4[celltype_col].value_counts())

(596486, 76)
pica_cell_type_complete_01_gdT_corrected
CD4 naive/Tcm    407516
CD4 Tem           96510
Tfh               45002
Treg              39431
CD4 cytoT          8027
Name: count, dtype: int64


In [ ]:
# Extract raw count matrix for pseudobulk aggregation
X = adata_cd4.layers["counts"]

# Convert sparse matrix to CSR format for efficient row indexing
if sparse.issparse(X):
    X = X.tocsr()

# Store gene names
genes = adata_cd4.var_names.astype(str)

# Get original integer positions of kept CD4 cells
cell_positions = np.where(keep_mask.values)[0]

print(X.shape)

(596486, 38606)


## Create Pseudobulk

In [21]:
# donor × cell type
obs_cd4["pseudobulk_id"] = (
    obs_cd4[donor_col].astype(str) + "__" + obs_cd4[celltype_col].astype(str)
)

# number of pseudobulk samples
print(obs_cd4["pseudobulk_id"].nunique())
print(obs_cd4["pseudobulk_id"].value_counts().describe())

639
count     639.000000
mean      933.467919
std      1308.386822
min         1.000000
25%       176.000000
50%       381.000000
75%      1012.000000
max      7531.000000
Name: count, dtype: float64


In [ ]:
# Aggregate raw counts
pb_counts = []
pb_meta = []

# Sum raw counts across cells belonging to the same donor and CD4 subset
for pb_id, group_index in obs_cd4.groupby("pseudobulk_id").indices.items():
    
    # Convert positions within obs_cd4 back to positions in the original AnnData object
    original_positions = cell_positions[group_index]
    meta_sub = obs_cd4.iloc[group_index]
    
    summed_counts = np.asarray(X[original_positions, :].sum(axis=0)).ravel()
    
    pb_counts.append(summed_counts)
    
    # Store sample-level metadata for downstream DESeq2 modelling
    pb_meta.append({
        "sample_id": pb_id,
        "donor_id": meta_sub[donor_col].iloc[0],
        "cell_type": meta_sub[celltype_col].iloc[0],
        "age": meta_sub[age_col].iloc[0],
        "sex": meta_sub[sex_col].iloc[0],
        "batch": meta_sub[batch_col].iloc[0],
        "n_cells": len(original_positions)
    })

# Create pseudobulk count matrix and metadata table
pb_counts_df = pd.DataFrame(
    pb_counts,
    index=[m["sample_id"] for m in pb_meta],
    columns=genes
)

pb_meta_df = pd.DataFrame(pb_meta).set_index("sample_id")

print("Pseudobulk count matrix:")
print(pb_counts_df.shape)

print("\nPseudobulk metadata:")
print(pb_meta_df.shape)

print("\nPseudobulk samples per cell type:")
print(pb_meta_df["cell_type"].value_counts())

Pseudobulk count matrix:
(639, 38606)

Pseudobulk metadata:
(639, 6)

Pseudobulk samples per cell type:
cell_type
CD4 Tem          128
CD4 naive/Tcm    128
Tfh              128
Treg             128
CD4 cytoT        127
Name: count, dtype: int64


In [ ]:
# Identify donors missing CD4 cytoT pseudobulk samples
all_donors = set(pb_meta_df["donor_id"].unique())

cytot_donors = set(
    pb_meta_df.loc[pb_meta_df["cell_type"] == "CD4 cytoT", "donor_id"]
)

missing_cytot_donors = all_donors - cytot_donors

missing_cytot_donors

{'PICA0006'}

In [ ]:
# Inspect CD4 subset composition for donors without CD4 cytoT samples
obs_cd4.loc[
    obs_cd4["pica_id"].isin(missing_cytot_donors),
    celltype_col
].value_counts()

pica_cell_type_complete_01_gdT_corrected
CD4 naive/Tcm    1456
Treg               64
CD4 Tem            46
Tfh                10
CD4 cytoT           0
Name: count, dtype: int64

In [ ]:
# Remove pseudobulk samples generated from too few cells
min_cells = 20

keep_samples = pb_meta_df["n_cells"] >= min_cells

pb_counts_df = pb_counts_df.loc[keep_samples]
pb_meta_df = pb_meta_df.loc[keep_samples]

print("After filtering pseudobulk samples with n_cells <", min_cells)
print("Counts:", pb_counts_df.shape)
print("Metadata:", pb_meta_df.shape)

print("\nSamples per cell type after filtering:")
print(pb_meta_df["cell_type"].value_counts())

print("\nCell counts per cell type after filtering:")
print(pb_meta_df.groupby("cell_type")["n_cells"].sum())

After filtering pseudobulk samples with n_cells < 20
Counts: (594, 38606)
Metadata: (594, 6)

Samples per cell type after filtering:
cell_type
CD4 naive/Tcm    128
CD4 Tem          127
Treg             127
Tfh              126
CD4 cytoT         86
Name: count, dtype: int64

Cell counts per cell type after filtering:
cell_type
CD4 Tem           96493
CD4 cytoT          7613
CD4 naive/Tcm    407516
Tfh               44978
Treg              39429
Name: n_cells, dtype: int64


In [33]:
pb_counts_df.T.to_csv("/scratch/user/s4575250/BIOX7014_Thesis/write/04_DEG/PICA_Batch001-Batch007/CD4_pseudobulk_counts_gene_by_sample.csv")
pb_meta_df.to_csv("/scratch/user/s4575250/BIOX7014_Thesis/write/04_DEG/PICA_Batch001-Batch007/CD4_pseudobulk_metadata.csv")

In [35]:
# Sanity checks
counts_check = pd.read_csv("/scratch/user/s4575250/BIOX7014_Thesis/write/04_DEG/PICA_Batch001-Batch007/CD4_pseudobulk_counts_gene_by_sample.csv", index_col=0)
meta_check = pd.read_csv("/scratch/user/s4575250/BIOX7014_Thesis/write/04_DEG/PICA_Batch001-Batch007/CD4_pseudobulk_metadata.csv", index_col=0)

print("Counts check:", counts_check.shape)
print("Metadata check:", meta_check.shape)

assert list(counts_check.columns) == list(meta_check.index)

print("Sample names match between counts and metadata.")
print(meta_check.head())

Counts check: (38606, 594)
Metadata check: (594, 6)
Sample names match between counts and metadata.
                         donor_id      cell_type   age   sex  \
sample_id                                                      
PICA0001__CD4 Tem        PICA0001        CD4 Tem  2.09  Male   
PICA0001__CD4 cytoT      PICA0001      CD4 cytoT  2.09  Male   
PICA0001__CD4 naive/Tcm  PICA0001  CD4 naive/Tcm  2.09  Male   
PICA0001__Tfh            PICA0001            Tfh  2.09  Male   
PICA0001__Treg           PICA0001           Treg  2.09  Male   

                                                              batch  n_cells  
sample_id                                                                     
PICA0001__CD4 Tem        20240530_WGS_20240530_sc_PICA0001-PICA0007       33  
PICA0001__CD4 cytoT      20240530_WGS_20240530_sc_PICA0001-PICA0007       33  
PICA0001__CD4 naive/Tcm  20240530_WGS_20240530_sc_PICA0001-PICA0007     2243  
PICA0001__Tfh            20240530_WGS_20240530_sc_PICA00